# RAG with HuggingFace and Milvus - Student Notebook

In this assignment, you will implement a complete RAG (Retrieval-Augmented Generation) pipeline using:
- **Dataset**: HuggingFace Documentation (`m-ric/huggingface_doc`)
- **Vector Store**: Milvus
- **Embeddings**: BGE-small-en-v1.5
- **LLM**: Microsoft Phi-3-mini-4k-instruct/"Qwen/Qwen2-1.5B-Instruct"
- **Evaluation**: Opik (AnswerRelevance, Hallucination)

## Instructions
1. Read through each section carefully
2. Complete the code in cells marked with `# TODO`
3. Run all cells in order
4. Verify your implementation with the evaluation cells


---

##https://github.com/milvus-io/milvus

## 1. Setup

Install required dependencies and configure environment.

In [1]:
# Install dependencies
!pip install -q pymilvus sentence-transformers datasets torch accelerate opik tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.7/159.7 kB 8.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.7/417.7 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.9/71.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.4/27.4 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13

In [2]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 144.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1


In [ ]:
import os
import json
from typing import List, Dict, Tuple
from tqdm import tqdm

# Set your HuggingFace token for model access
# You can get one at: https://huggingface.co/settings/tokens
os.environ["HF_TOKEN"] = "YOUR_HUGGINGFACE_TOKEN"  # Replace with your token

# Opik configuration (optional - for generation evaluation)
# Get your API key at: https://www.comet.com/
os.environ["OPIK_API_KEY"] = "YOUR_COMET_TOKEN"  # Replace with your Opik API key if available

print("Environment configured!")

Environment configured!


## 2. Data Loading

Load the HuggingFace documentation dataset.

In [4]:
from datasets import load_dataset

# Load the HuggingFace documentation dataset
dataset = load_dataset("m-ric/huggingface_doc", split="train")

print(f"Dataset loaded with {len(dataset)} documents")
print(f"Columns: {dataset.column_names}")
print(f"\nSample document (first 500 chars):")
print(dataset[0]["text"][:500])

README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

huggingface_doc.csv: reconstructing file:   0%|          |  0.00B / 22.0MB            

huggingface_doc.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2647 [00:00<?, ? examples/s]

Dataset loaded with 2647 documents
Columns: ['text', 'source']

Sample document (first 500 chars):
 Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. 

## 1. Enter the Hugging Face Repository ID and your desired endpoint name:

<img src="https://raw.githubusercontent.com/huggingface/hf-endpoints-docu


In [5]:
# Extract text and source information
documents = []
for item in dataset:
    documents.append({
        "text": item["text"],
        "source": item["source"]
    })

print(f"Extracted {len(documents)} documents")

# For this assignment, we'll use a subset to keep things manageable
MAX_DOCS = 500
documents = documents[:MAX_DOCS]
print(f"Using {len(documents)} documents for this assignment")

Extracted 2647 documents
Using 500 documents for this assignment


## 3. Chunking

Split documents into smaller chunks for better retrieval.

### Your Task
Implement the `chunk_document` function that:
1. Takes a text string, chunk_size, and chunk_overlap as parameters
2. Splits the text into overlapping chunks of the specified size
3. Returns a list of chunk strings

### Hints
- Use a sliding window approach with step = chunk_size - chunk_overlap
- Handle edge cases: empty text, text shorter than chunk_size
- Make sure each chunk is non-empty before adding it

In [6]:
# ============================================================
# TODO: IMPLEMENT CHUNKING (15 points)
# ============================================================

def chunk_document(text: str, chunk_size: int = 1000, chunk_overlap: int = 200) -> List[str]:
    """
    Split a document into overlapping chunks of fixed size.

    Args:
        text: The document text to chunk
        chunk_size: Maximum size of each chunk in characters
        chunk_overlap: Number of overlapping characters between chunks

    Returns:
        List of text chunks
    """
    chunks = []

    # TODO: Implement fixed-size chunking with overlap
    #
    # Step 1: Handle edge cases (empty text, short text)
    # Hint: If text is empty or shorter than chunk_size, return it as a single chunk

    # Step 2: Calculate step size for sliding window
    # Hint: step = chunk_size - chunk_overlap

    # Step 3: Create chunks using a while loop
    # Hint: Start at position 0, extract chunk_size characters,
    #       then move forward by step size

    # Step 4: Only add non-empty chunks (use .strip() to check)

    # YOUR CODE HERE
    # Step 1: Handle edge cases (empty text, short text)
    if not text or not text.strip():
        return []
    if len(text) <= chunk_size:
        return [text]

    # Step 2: Calculate step size for sliding window
    step = chunk_size - chunk_overlap

    # Ensure step is strictly positive to prevent infinite loops
    if step <= 0:
        raise ValueError("chunk_overlap must be strictly less than chunk_size")

    # Step 3: Create chunks using a while loop
    current_pos = 0
    while current_pos < len(text):
        chunk = text[current_pos : current_pos + chunk_size]

        # Step 4: Only add non-empty chunks (use .strip() to check)
        if chunk.strip():
            chunks.append(chunk)

        current_pos += step

    return chunks


def chunk_all_documents(documents: List[Dict], chunk_size: int = 1000, chunk_overlap: int = 200) -> List[Dict]:
    """
    Chunk all documents and preserve metadata.

    Args:
        documents: List of document dicts with 'text' and 'source' keys
        chunk_size: Maximum chunk size
        chunk_overlap: Overlap between chunks

    Returns:
        List of chunk dicts with 'text', 'source', and 'chunk_id' keys
    """
    all_chunks = []
    chunk_id = 0

    # TODO: Iterate through documents, chunk each one, and add metadata
    #
    # For each document:
    #   1. Get text and source from document dict
    #   2. Call chunk_document() to get chunks
    #   3. For each chunk, create a dict with chunk_id, text, and source
    #   4. Append to all_chunks and increment chunk_id

    # YOUR CODE HERE
    for doc in documents:
        text = doc['text']
        source = doc['source']
        chunks = chunk_document(text, chunk_size, chunk_overlap)
        for chunk_text in chunks:
            all_chunks.append({
                'chunk_id': chunk_id,
                'text': chunk_text,
                'source': source
            })
            chunk_id += 1

    return all_chunks

In [7]:
# Test your chunking implementation
test_text = "A" * 2500  # 2500 characters
test_chunks = chunk_document(test_text, chunk_size=1000, chunk_overlap=200)

print(f"Test: 2500 char text with chunk_size=1000, overlap=200")
print(f"Expected chunks: ~4")
print(f"Your chunks: {len(test_chunks)}")

if len(test_chunks) >= 3 and len(test_chunks) <= 5:
    print("✅ Chunking test passed!")
else:
    print("❌ Check your chunking implementation")

Test: 2500 char text with chunk_size=1000, overlap=200
Expected chunks: ~4
Your chunks: 4
✅ Chunking test passed!


In [8]:
# Create chunks from all documents
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

chunks = chunk_all_documents(documents, CHUNK_SIZE, CHUNK_OVERLAP)

print(f"\nCreated {len(chunks)} chunks from {len(documents)} documents")
print(f"Average chunks per document: {len(chunks) / len(documents):.2f}")

# Show sample chunk
if chunks:
    print(f"\nSample chunk:")
    print(f"  ID: {chunks[0]['chunk_id']}")
    print(f"  Source: {chunks[0]['source']}")
    print(f"  Text (first 200 chars): {chunks[0]['text'][:200]}...")


Created 5651 chunks from 500 documents
Average chunks per document: 11.30

Sample chunk:
  ID: 0
  Source: huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx
  Text (first 200 chars):  Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deplo...


## 4. Embeddings

Generate vector embeddings for each chunk using BGE-small-en-v1.5.

### Your Task
Implement the `generate_embeddings` function that:
1. Processes texts in batches for memory efficiency
2. Uses the SentenceTransformer model to generate embeddings
3. Returns embeddings as a list of lists (for Milvus compatibility)

### Hints
- Use `model.encode()` with `normalize_embeddings=True` for cosine similarity
- Process in batches to avoid memory issues
- Convert numpy arrays to lists using `.tolist()`

In [9]:
from sentence_transformers import SentenceTransformer

# Load the embedding model
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5" #Use any model of your choice from Sentence Transformers
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

print(f"Loaded embedding model: {EMBEDDING_MODEL}")

# Test embedding
test_embedding = embedding_model.encode(["This is a test"], normalize_embeddings=True)
EMBEDDING_DIM = len(test_embedding[0])
print(f"Embedding dimension: {EMBEDDING_DIM}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded embedding model: BAAI/bge-small-en-v1.5
Embedding dimension: 384


In [10]:
# ============================================================
# TODO: IMPLEMENT EMBEDDING GENERATION (15 points)
# ============================================================

def generate_embeddings(texts: List[str], model: SentenceTransformer, batch_size: int = 32) -> List[List[float]]:
    """
    Generate embeddings for a list of texts.

    Args:
        texts: List of text strings to embed
        model: SentenceTransformer model
        batch_size: Number of texts to process at once

    Returns:
        List of embedding vectors (as lists of floats)
    """
    all_embeddings = []

    # TODO: Generate embeddings in batches
    #
    # Step 1: Loop through texts in batches of size batch_size
    # Hint: Use range(0, len(texts), batch_size) to get batch start indices

    # Step 2: For each batch, call model.encode() with:
    #   - The batch of texts
    #   - normalize_embeddings=True (important for cosine similarity)
    #   - show_progress_bar=False (we use tqdm at the outer level)

    # Step 3: Convert to list format and extend all_embeddings
    # Hint: Use .tolist() to convert numpy array to Python list

    # YOUR CODE HERE
    for i in tqdm(range(0, len(texts), batch_size), desc="Generating embeddings"):
        batch_texts = texts[i : i + batch_size]
        batch_embeddings = model.encode(batch_texts, normalize_embeddings=True, show_progress_bar=False)
        all_embeddings.extend(batch_embeddings.tolist())

    return all_embeddings

In [11]:
# Test your embedding generation
test_texts = ["Hello world", "This is a test", "RAG is cool"]
test_embeddings = generate_embeddings(test_texts, embedding_model)

print(f"Generated {len(test_embeddings)} embeddings")
print(f"Embedding dimension: {len(test_embeddings[0]) if test_embeddings else 0}")

if len(test_embeddings) == 3 and len(test_embeddings[0]) == 384:
    print("✅ Embedding generation test passed!")
else:
    print("❌ Check your embedding implementation")

Generating embeddings: 100%|██████████| 1/1 [00:00<00:00, 57.17it/s]

Generated 3 embeddings
Embedding dimension: 384
✅ Embedding generation test passed!


In [12]:
# Generate embeddings for all chunks
chunk_texts = [chunk["text"] for chunk in chunks]
embeddings = generate_embeddings(chunk_texts, embedding_model)

print(f"\nGenerated {len(embeddings)} embeddings")
if embeddings:
    print(f"Embedding dimension: {len(embeddings[0])}")
    print(f"Sample embedding (first 10 values): {embeddings[0][:10]}")

Generating embeddings: 100%|██████████| 177/177 [11:17<00:00,  3.83s/it]


Generated 5651 embeddings
Embedding dimension: 384
Sample embedding (first 10 values): [-0.07532963901758194, -0.027507975697517395, -0.039956074208021164, -0.04049215093255043, 0.033339984714984894, 0.042965129017829895, -0.043336279690265656, -0.04493827372789383, -0.05554315447807312, 0.026720337569713593]


## 5. Vector Store (Milvus)

Store embeddings in Milvus for efficient similarity search.

### Your Task
1. Implement `setup_milvus_collection` to create a new collection
2. Implement `insert_data_to_milvus` to insert chunks and embeddings

### Hints
- Use `client.has_collection()` to check if collection exists
- Use `client.drop_collection()` to remove existing collection
- Use `client.create_collection()` with dimension and metric_type parameters
- Use `client.insert()` to add data

In [13]:
pip install pymilvus[milvus_lite]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.6/269.6 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 142.1 MB/s eta 0:00:00


In [14]:
from pymilvus import MilvusClient

# Initialize Milvus client (uses Milvus Lite - stores data locally)
MILVUS_DB_PATH = "./hf_docs_milvus.db"
milvus_client = MilvusClient(uri=MILVUS_DB_PATH)

COLLECTION_NAME = "hf_documentation"

print(f"Milvus client initialized with database: {MILVUS_DB_PATH}")

Milvus client initialized with database: ./hf_docs_milvus.db


In [15]:
# ============================================================
# TODO: IMPLEMENT MILVUS COLLECTION SETUP (10 points)
# ============================================================

def setup_milvus_collection(client: MilvusClient, collection_name: str, embedding_dim: int):
    """
    Create a Milvus collection for storing document embeddings.

    Args:
        client: MilvusClient instance
        collection_name: Name of the collection to create
        embedding_dim: Dimension of the embedding vectors
    """
    # TODO: Create a Milvus collection
    #
    # Step 1: Check if collection already exists using client.has_collection()
    # Step 2: If exists, drop it using client.drop_collection()
    # Step 3: Create new collection using client.create_collection() with:
    #   - collection_name: the name parameter
    #   - dimension: embedding_dim parameter
    #   - metric_type: "IP" (Inner Product for cosine similarity)
    #   - consistency_level: "Strong"

    # YOUR CODE HERE
    if client.has_collection(collection_name):
        client.drop_collection(collection_name)

    client.create_collection(
        collection_name,
        dimension=embedding_dim,
        metric_type="IP",
        consistency_level="Strong"
    )

    print(f"Created collection: {collection_name} with dimension {embedding_dim}")

In [16]:
# Setup the collection
setup_milvus_collection(milvus_client, COLLECTION_NAME, EMBEDDING_DIM)

ERROR:grpc._server:Exception calling application: Method not implemented!
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/grpc/_server.py", line 608, in _call_behavior
    response_or_iterator = behavior(argument, context)
  File "/usr/local/lib/python3.13/dist-packages/pymilvus/grpc_gen/milvus_pb2_grpc.py", line 1357, in AllocTimestamp
    raise NotImplementedError('Method not implemented!')
NotImplementedError: Method not implemented!


Created collection: hf_documentation with dimension 384


In [17]:
# ============================================================
# TODO: IMPLEMENT DATA INSERTION (10 points)
# ============================================================

def insert_data_to_milvus(
    client: MilvusClient,
    collection_name: str,
    chunks: List[Dict],
    embeddings: List[List[float]],
    batch_size: int = 100
):
    """
    Insert document chunks and embeddings into Milvus.

    Args:
        client: MilvusClient instance
        collection_name: Name of the collection
        chunks: List of chunk dictionaries with text and metadata
        embeddings: List of embedding vectors
        batch_size: Number of records to insert at once

    Returns:
        Total number of inserted records
    """
    total_inserted = 0

    # TODO: Insert data into Milvus
    #
    # Step 1: Prepare data as a list of dictionaries, where each dict has:
    #   - "id": chunk["chunk_id"]
    #   - "vector": the corresponding embedding
    #   - "text": chunk["text"]
    #   - "source": chunk["source"]

    # Step 2: Insert in batches using client.insert()
    # Hint: Loop through data in batches and call:
    #   result = client.insert(collection_name=collection_name, data=batch)
    #   total_inserted += result["insert_count"]

    # YOUR CODE HERE
    data = [
        {
            "id": chunk["chunk_id"],
            "vector": embedding,
            "text": chunk["text"],
            "source": chunk["source"]
        }
        for chunk, embedding in zip(chunks, embeddings)
    ]

    for i in tqdm(range(0, len(data), batch_size), desc="Inserting data"):
        batch = data[i : i + batch_size]
        result = client.insert(collection_name=collection_name, data=batch)
        total_inserted += result["insert_count"]

    return total_inserted

In [18]:
# Insert data into Milvus
inserted_count = insert_data_to_milvus(milvus_client, COLLECTION_NAME, chunks, embeddings)

print(f"\nInserted {inserted_count} records into Milvus")

if inserted_count == len(chunks):
    print("✅ All chunks inserted successfully!")
else:
    print("❌ Not all chunks were inserted. Check your implementation.")

Inserting data: 100%|██████████| 57/57 [00:00<00:00, 81.96it/s]


Inserted 5651 records into Milvus
✅ All chunks inserted successfully!


## 6. Retrieval

Implement semantic search to retrieve relevant documents for a query.

### Your Task
Implement the `retrieve_documents` function that:
1. Generates an embedding for the query
2. Searches Milvus for similar vectors
3. Returns the top-K most relevant documents

### Hints
- Use `embedding_model.encode()` to embed the query
- Use `client.search()` to find similar vectors
- Extract text and source from the search results

In [19]:
# ============================================================
# TODO: IMPLEMENT RETRIEVAL (25 points)
# ============================================================

def retrieve_documents(
    query: str,
    client: MilvusClient,
    collection_name: str,
    embedding_model: SentenceTransformer,
    top_k: int = 5
) -> List[Dict]:
    """
    Retrieve the most relevant documents for a query.

    Args:
        query: The search query
        client: MilvusClient instance
        collection_name: Name of the collection to search
        embedding_model: Model to generate query embedding
        top_k: Number of results to return

    Returns:
        List of dictionaries with 'text', 'source', and 'score' keys
    """
    # TODO: Implement semantic search
    #
    # Step 1: Generate embedding for the query
    # Hint: Use embedding_model.encode([query], normalize_embeddings=True)
    #       Then convert to list: .tolist()[0]

    # Step 2: Search in Milvus using client.search()
    # Required parameters:
    #   - collection_name: collection_name
    #   - data: [query_embedding] (list containing the embedding)
    #   - limit: top_k
    #   - search_params: {"metric_type": "IP", "params": {}}
    #   - output_fields: ["text", "source"]

    # Step 3: Format results as list of dicts
    # Each dict should have:
    #   - "text": result["entity"]["text"]
    #   - "source": result["entity"]["source"]
    #   - "score": result["distance"]

    # YOUR CODE HERE
    retrieved_docs = []
    query_embedding = embedding_model.encode([query], normalize_embeddings=True).tolist()[0]
    search_results = client.search(
        collection_name=collection_name,
        data=[query_embedding],
        limit=top_k,
        search_params={"metric_type": "IP", "params": {}},
        output_fields=["text", "source"]
    )

    for result in search_results[0]:
        retrieved_docs.append({
            "text": result["entity"]["text"],
            "source": result["entity"]["source"],
            "score": result["distance"]
        })



    return retrieved_docs

In [20]:
# Test retrieval
test_query = "How do I fine-tune a transformer model?"

retrieved = retrieve_documents(
    query=test_query,
    client=milvus_client,
    collection_name=COLLECTION_NAME,
    embedding_model=embedding_model,
    top_k=3
)

print(f"Query: {test_query}")
print(f"\nRetrieved {len(retrieved)} documents:")
for i, doc in enumerate(retrieved):
    print(f"\n--- Document {i+1} (Score: {doc.get('score', 'N/A')}) ---")
    print(f"Source: {doc.get('source', 'N/A')}")
    print(f"Text: {doc.get('text', 'N/A')[:300]}...")

if len(retrieved) == 3 and all('text' in d for d in retrieved):
    print("\n✅ Retrieval test passed!")
else:
    print("\n❌ Check your retrieval implementation")

Query: How do I fine-tune a transformer model?

Retrieved 3 documents:

--- Document 1 (Score: 0.8237407207489014) ---
Source: huggingface/blog/blob/main/vision_language_pretraining.md
Text:  models from Transformers.*

...

--- Document 2 (Score: 0.7484297752380371) ---
Source: huggingface/blog/blob/main/ray-rag.md
Text: ects/rag/finetune_rag_ray.sh) for faster distributed fine-tuning, you can leverage RAG for retrieval-based generation on your own knowledge-intensive tasks.


Also, hyperparameter tuning is another aspect of transformer fine tuning and can have [huge impacts on accuracy](https://medium.com/distribut...

--- Document 3 (Score: 0.7302744388580322) ---
Source: huggingface/blog/blob/main/lewis-tunstall-interview.md
Text: n try to integrate it into your application. 

So what I've been working on for the last few months on the transformers library is providing the functionality to export these models into a format that lets you run them much more efficiently using tools th

## 7. Generation

Generate answers using Microsoft Phi-3-mini-4k-instruct/Qwen.

### Your Task
Implement the `generate_answer` function that:
1. Combines retrieved documents into a context string
2. Formats the prompt using the provided template
3. Generates an answer using the language model
4. Returns a structured result dictionary

### Hints
- Join document texts with newlines to create context
- Use the PROMPT_TEMPLATE.format() to fill in context and question
- Call the generator pipeline with appropriate parameters

### https://huggingface.co/microsoft/Phi-3-mini-4k-instruct

### https://huggingface.co/microsoft/Phi-3.5-mini-instruct

### https://huggingface.co/Qwen/Qwen2-1.5B-Instruct

### FEEL FREE TO USE A PROPRIETARY MODEL LIKE OPENAI, CLAUDE

In [21]:
!pip install accelerate bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 36.4 MB/s eta 0:00:00


In [22]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
import torch

# Load the language model
LLM_MODEL = "Qwen/Qwen2-1.5B-Instruct"

print(f"Loading model: {LLM_MODEL}")
print("This may take a few minutes...")

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

# Configure 4-bit quantization for minimal memory usage
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",               # Uses Normalized Float 4 (best for LLMs)
    bnb_4bit_compute_dtype=torch.float16,    # Computes in fp16 to maintain speed
    bnb_4bit_use_double_quant=True           # Saves an additional ~0.4 bits per parameter
)

# Load the model with quantization and optimized memory mapping
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    quantization_config=bnb_config,
    device_map="auto",          # Automatically manages GPU/CPU memory placement
    low_cpu_mem_usage=True,     # Prevents system RAM from spiking during model load
    trust_remote_code=False
)

# Create text generation pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

print("Model loaded successfully!")


Loading model: Qwen/Qwen2-1.5B-Instruct
This may take a few minutes...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


### MODIFY THIS TO SUIT YOUR MODEL

In [23]:
# Prompt template for RAG (YOU ARE FREE TO MODIFY)
PROMPT_TEMPLATE = """Use the following pieces of information enclosed in <context> tags to provide an answer to the question enclosed in <question> tags.
If the context doesn't contain enough information to answer the question, say "I don't have enough information to answer this question."

<context>
{context}
</context>

<question>
{question}
</question>

Answer:"""

In [24]:
# ============================================================
# TODO: IMPLEMENT GENERATION (25 points)
# ============================================================

def generate_answer(
    query: str,
    retrieved_docs: List[Dict],
    generator: pipeline,
    max_new_tokens: int = 256
) -> Dict:
    """
    Generate an answer using retrieved documents as context.

    Args:
        query: The user's question
        retrieved_docs: List of retrieved document dictionaries
        generator: HuggingFace text generation pipeline
        max_new_tokens: Maximum tokens to generate

    Returns:
        Dictionary with 'answer', 'context', 'query', and 'retrieved_docs'
    """
    # TODO: Generate an answer using the RAG pattern
    #
    # Step 1: Combine retrieved documents into context
    # Hint: Join doc["text"] for each doc with "\n\n" separator

    # Step 2: Format the prompt using PROMPT_TEMPLATE
    # Hint: prompt = PROMPT_TEMPLATE.format(context=context, question=query)

    # Step 3: Generate response using the generator pipeline
    # Call generator() with:
    #   - prompt (first argument)
    #   - max_new_tokens=max_new_tokens
    #   - do_sample=True
    #   - temperature=0.7
    #   - top_p=0.9
    #   - return_full_text=False

    # Step 4: Extract the generated text
    # Hint: outputs[0]["generated_text"].strip()

    # Step 5: Return result dictionary

    # YOUR CODE HERE
    context = ""
    answer = ""

    for doc in retrieved_docs:
        context += doc["text"] + "\n\n"

    prompt = PROMPT_TEMPLATE.format(context=context, question=query)

    outputs = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        return_full_text=False
    )

    answer = outputs[0]["generated_text"].strip()

    return {
        "query": query,
        "answer": answer,
        "context": context,
        "retrieved_docs": retrieved_docs
    }

In [25]:
# Test generation
test_query = "How do I fine-tune a transformer model?"

# Retrieve relevant documents
retrieved = retrieve_documents(
    query=test_query,
    client=milvus_client,
    collection_name=COLLECTION_NAME,
    embedding_model=embedding_model,
    top_k=3
)

# Generate answer
result = generate_answer(
    query=test_query,
    retrieved_docs=retrieved,
    generator=generator
)

print(f"Question: {result['query']}")
print(f"\nAnswer: {result['answer']}")

if result['answer'] and len(result['answer']) > 10:
    print("\n✅ Generation test passed!")
else:
    print("\n❌ Check your generation implementation")

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'top_p', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Question: How do I fine-tune a transformer model?

Answer: Use the finetune_rag_ray.sh script for faster distributed fine-tuning. Also, hyperparameter tuning is another aspect of transformer fine-tuning and can have huge impacts on accuracy. By using Ray Tune's integration with PyTorch Lightning or the built-in integration with Huggingface transformers, you can run experiments to find the perfect hyperparameters for your RAG model. And lastly, stay tuned for a potential Tensorflow implementation of RAG integrating it into your application. The transformers library provides the functionality to export models into a format that allows efficient execution using tools in the open-source ecosystem such as ONNX. This includes converting PyTorch models to TensorFlow or other hardware-specific formats. Additionally, the library takes care of the conversion process by running one line of code. 

To fine-tune a transformer model, you can use the `finetune_rag_ray.sh` script provided by the trans

In [26]:
# Complete RAG pipeline function (DO NOT MODIFY)

def rag_query(
    query: str,
    client: MilvusClient,
    collection_name: str,
    embedding_model: SentenceTransformer,
    generator: pipeline,
    top_k: int = 5,
    max_new_tokens: int = 256
) -> Dict:
    """
    Complete RAG pipeline: retrieve then generate.
    """
    # Retrieve
    retrieved_docs = retrieve_documents(
        query=query,
        client=client,
        collection_name=collection_name,
        embedding_model=embedding_model,
        top_k=top_k
    )

    # Generate
    result = generate_answer(
        query=query,
        retrieved_docs=retrieved_docs,
        generator=generator,
        max_new_tokens=max_new_tokens
    )

    return result

In [27]:
# Test complete pipeline with multiple queries
test_queries = [
    "What is the Trainer class in transformers?",
    "How do I load a dataset from HuggingFace?",
    "What is Gradio used for?"
]

for query in test_queries:
    print(f"\n{'='*60}")
    result = rag_query(
        query=query,
        client=milvus_client,
        collection_name=COLLECTION_NAME,
        embedding_model=embedding_model,
        generator=generator,
        top_k=3
    )
    print(f"Q: {result['query']}")
    print(f"A: {result['answer']}")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is the Trainer class in transformers?
A: The Trainer class in transformers is a class that provides methods for training and evaluating a pre-trained model. It takes into account the configuration and data used during the training process and produces a report of the performance of the model on the evaluation dataset. It uses the same interface as the base class, but it's designed to work with the specific tasks (e.g., classification, regression, etc.) supported by transformers. Additionally, it supports multiple metrics like accuracy, precision, recall, and F1 score which are calculated using different functions provided within the library.

The Trainer class in transformers offers several useful functionalities including:

- Instantiating a Trainer object using the provided arguments and model.
  
- Updating and returning a dictionary containing the per-class accuracy values.
  
- Creating a model card with additional information based on the Trainer output.
  
- Training the

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How do I load a dataset from HuggingFace?
A: You can use the load_dataset function provided by the Hugging Face Datasets library. To load a local CSV file, you need to provide the filename as well as a data_files argument pointing to a filepath or URL. Once loaded, you can interact with the dataset by accessing splits and elements through indexing. 

For datasets hosted on GitHub or the UCI Machine Learning Repository, you can use the same function but with different arguments specific to those repositories. For example, if loading a dataset from the UCI Machine Learning Repository, you would provide the repository name, e.g., uci-machine-learning-databases.

It's also important to note that everything is saved to disk using Apache Arrow, making the process efficient and reproducible.

To try it out yourself, pick another dataset from GitHub or the UCI Machine Learning Repository and follow the instructions above. For bonus points, consider loading a dataset that's stored in a CSV o